In [0]:
customers_df = spark.read.table("retail_project.silver.silver_customers")
orders_df = spark.read.table("retail_project.silver.silver_orders")

customers_df.display()
orders_df.display()

In [0]:
dim_customer = customers_df.select("customer_id",
                    "customer_name",
                    "email",
                    "city",
                    "state",
                    "loyalty_points",
                    "status"                    
                    )
dim_customer.display()

In [0]:
spark.sql('CREATE SCHEMA IF NOT EXISTS retail_project.gold')
dim_customer.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.gold_dim_customer')    

In [0]:
fact_sales = orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "left"
).select(
    orders_df.order_id,
    orders_df.order_date,
    orders_df.customer_id,
    customers_df.customer_name,
    orders_df.product_id,
    orders_df.product_name,
    orders_df.category,
    orders_df.quantity,
    orders_df.unit_price,
    orders_df.discount,
    orders_df.total_amount,
    customers_df.city,
    customers_df.state,
    customers_df.gender,
    customers_df.loyalty_points
)

# fact_sales.display()


from pyspark.sql.functions import year , month , dayofmonth

order_enriched = fact_sales.\
    withColumn("Year",year("order_date")).\
    withColumn("Month",month("order_date")).\
    withColumn("Day",dayofmonth("order_date"))

order_enriched.display()

In [0]:
from pyspark.sql.functions import sum, countDistinct, avg, min,max

gold_df = order_enriched.groupBy(
    "product_id",
    "product_name",
    "customer_id",
    "customer_name",
    "category",
    "Year",
    "Month",
    "order_date",
    "gender",
    "city",
    "state",
    "loyalty_points"
).agg(
    sum("total_amount").alias("Total_revenue"),
    sum("quantity").alias("Total_quantity"),
    countDistinct("order_id").alias("Total_orders"),
    avg("total_amount").alias("Avg_revenue"),
    avg("quantity").alias("Avg_quantity"),
    avg("loyalty_points").alias("Avg_loyalty_points"),
    min("unit_price").alias("Min_price"),
    max("unit_price").alias("Max_price")
)
gold_df.display()
     


In [0]:
gold_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.gold_fact_Sales')

In [0]:
%sql
select count(*) from retail_project.gold.gold_fact_Sales

In [0]:
# ======================================================================
# SCD TYPE 2 – CUSTOMER DIMENSION (manual approach with snapshot_id)
# ======================================================================
# This cell implements Slowly Changing Dimension Type 2 (SCD2) for the
# customer dimension.  SCD2 keeps a FULL HISTORY of every change:
#   • When a customer's attributes change, the OLD row is "expired"
#     (is_current → False, effective_end_date → now).
#   • A NEW row is inserted with the updated attributes
#     (is_current → True, effective_end_date → 9999-12-31).
# This lets downstream fact tables do point-in-time joins.
# ======================================================================

# ── PART 1: Imports ─────────────────────────────────────────────────
# col / lit          → reference columns & literal values
# sha2, concat_ws    → build a hash fingerprint of each row's attributes
# coalesce           → turn NULLs into empty strings before hashing
# current_timestamp  → single batch timestamp used as snapshot_id
# monotonically_increasing_id → generate unique surrogate keys
from pyspark.sql.functions import (
    col, lit, sha2, concat_ws, coalesce, current_timestamp,
    when, monotonically_increasing_id
)
from pyspark.sql.types import TimestampType


# ── PART 2: Load Silver (source) tables ────────────────────────────
# These are the cleaned tables from the Silver layer.
customers_src = spark.read.table("retail_project.silver.silver_customers")
orders_src    = spark.read.table("retail_project.silver.silver_orders")


# ── PART 3: Create a snapshot_id ───────────────────────────────────
# snapshot_id = the same timestamp for EVERY row written in this batch.
# It tags the load so you can audit "what changed in which run".
snapshot_id = current_timestamp()


# ── PART 4: Build an attribute hash on the source ──────────────────
# We concatenate all the Type-2-tracked columns with "||" as separator,
# then compute a SHA-256 hash.  If ANY tracked attribute changes, the
# hash changes → we know the customer record was modified.
customers_src = customers_src.withColumn(
    "attr_hash",
    sha2(concat_ws("||",
        coalesce(col("customer_name"), lit("")),
        coalesce(col("email"), lit("")),
        coalesce(col("gender"), lit("")),
        coalesce(col("city"), lit("")),
        coalesce(col("state"), lit("")),
        coalesce(col("loyalty_points").cast("string"), lit("")),
        coalesce(col("status"), lit(""))
    ), 256)
)


# ── PART 5: Create the Gold SCD2 target table (if first run) ──────
# The table has:
#   surrogate_key        → unique ID for each VERSION of a customer
#   customer_id          → the natural / business key
#   <attributes>         → the tracked columns
#   attr_hash            → fingerprint for quick change detection
#   effective_start_date → when this version became active
#   effective_end_date   → when it was expired (9999-12-31 = still active)
#   is_current           → True for the latest version of each customer
#   snapshot_id          → which batch load created this row
spark.sql("""
CREATE TABLE IF NOT EXISTS retail_project.gold.gold_dim_customer_scd2_v2 (
    surrogate_key       BIGINT,
    customer_id         STRING,
    customer_name       STRING,
    email               STRING,
    gender              STRING,
    city                STRING,
    state               STRING,
    loyalty_points      INT,
    status              STRING,
    attr_hash           STRING,
    effective_start_date TIMESTAMP,
    effective_end_date   TIMESTAMP,
    is_current           BOOLEAN,
    snapshot_id          TIMESTAMP
) USING DELTA
""")


# ── PART 6: Read the existing dimension & find the latest version ──
# existing_dim = ALL rows (current + expired).
# current_dim  = only rows where is_current == True (one per customer).
existing_dim = spark.read.table("retail_project.gold.gold_dim_customer_scd2_v2")
current_dim  = existing_dim.filter(col("is_current") == True)


# ── PART 7: Determine the next surrogate key ──────────────────────
# We need a monotonically increasing surrogate key.  Find the current
# max and start new keys from max + 1.
max_sk = existing_dim.agg({"surrogate_key": "max"}).collect()[0][0]
next_sk_start = (max_sk or 0) + 1   # if table is empty, max_sk is None → start at 1


# ── PART 8: Identify NEW customers ────────────────────────────────
# left_anti join: keep source rows whose customer_id does NOT exist
# in the current dimension at all.  These are brand-new customers.
new_customers = customers_src.join(current_dim, "customer_id", "left_anti")


# ── PART 9: Identify CHANGED customers ────────────────────────────
# Inner join on customer_id, then filter where the hashes differ.
# These customers exist in the dimension but their attributes changed.
changed_customers = (
    customers_src.alias("src")
    .join(current_dim.alias("dim"), "customer_id", "inner")
    .where(col("src.attr_hash") != col("dim.attr_hash"))
    .select("src.*")
)


# ── PART 10: Union the rows that need to be INSERTED ──────────────
# Both new AND changed customers get a fresh row in the dimension.
insert_df = new_customers.unionByName(changed_customers)

# Add the SCD2 metadata columns:
#   • surrogate_key        → unique per-version key
#   • effective_start_date → this batch's timestamp
#   • effective_end_date   → far-future sentinel (still active)
#   • is_current           → True (this is the newest version)
#   • snapshot_id          → same batch timestamp for auditing
insert_df = (
    insert_df
    .withColumn("row_num", monotonically_increasing_id())
    .withColumn("surrogate_key", col("row_num") + lit(next_sk_start))
    .drop("row_num")
    .withColumn("effective_start_date", snapshot_id)
    .withColumn("effective_end_date", lit("9999-12-31 23:59:59").cast(TimestampType()))
    .withColumn("is_current", lit(True))
    .withColumn("snapshot_id", snapshot_id)
)


# ── PART 11: Expire old versions of CHANGED customers ─────────────
# For customers whose attributes changed, we must UPDATE the existing
# current row → set is_current = False and effective_end_date = now.
# This "closes" the old version's validity window.
changed_ids = [r.customer_id for r in changed_customers.select("customer_id").collect()]

if changed_ids:
    from delta.tables import DeltaTable
    delta_dim = DeltaTable.forName(spark, "retail_project.gold.gold_dim_customer_scd2_v2")

    delta_dim.update(
        condition=(col("is_current") == True) & (col("customer_id").isin(changed_ids)),
        set={
            "is_current": lit(False),
            "effective_end_date": snapshot_id
        }
    )


# ── PART 12: Append the new / changed-version rows ────────────────
# Only write if there is something to insert (new or changed customers).
if insert_df.count() > 0:
    insert_df.select(
        "surrogate_key", "customer_id", "customer_name", "email", "gender",
        "city", "state", "loyalty_points", "status", "attr_hash",
        "effective_start_date", "effective_end_date", "is_current", "snapshot_id"
    ).write.format("delta").mode("append").saveAsTable(
        "retail_project.gold.gold_dim_customer_scd2_v2"
    )

print("✅ SCD Type 2 dimension loaded with snapshot_id =", snapshot_id)


# ── PART 13: Display the full dimension ordered by history ─────────
# You should see:
#   • One row per customer if nothing changed (is_current = True)
#   • Multiple rows per customer if attributes changed over time,
#     with only the latest row having is_current = True.
result = spark.read.table("retail_project.gold.gold_dim_customer_scd2_v2")
result.orderBy("customer_id", "effective_start_date").display()